In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
import json
import shutil 
from tqdm import tqdm
import matplotlib.pyplot as plt
from plotly import graph_objects as go
import plotly.subplots as psu
import pandas as pd
from typing import Optional
import os

import waffles
from waffles.input_output.hdf5_structured import load_structured_waveformset
from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.BasicWfAna import BasicWfAna
from waffles.data_classes.IPDict import IPDict
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.baseline.baseline import SBaseline
from waffles.utils.numerical_utils import average_wf_ch
from waffles.utils.utils import compute_peaks_rise_fall_ch
from waffles.utils.selector_waveforms import WaveformSelector
from waffles.np02_utils.AutoMap import generate_ChannelMap, ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, dict_uniqch_to_module, dict_module_to_uniqch, strUch, ordered_channels_membrane
from waffles.np02_utils.PlotUtils import np02_gen_grids, plot_grid, plot_detectors, genhist, fithist, runBasicWfAnaNP02, plot_averages
from waffles.np02_utils.load_utils import open_processed, ch_read_calib

In [ ]:
#dettype = "membrane"
dettype = "cathode"
dettype = "pmt"

datadir = f"/eos/experiment/neutplatform/protodune/experiments/ProtoDUNE-VD/commissioning/"
endpoint = 106 if dettype == "cathode" else 107 if dettype == "membrane" else 110
listmods = ordered_modules_cathode if endpoint == 106 else ordered_modules_membrane if endpoint == 107 else ordered_modules_pmt

n = len(listmods)
group1 = listmods[:n//4]
group2 = listmods[n//4:n//2]
group3 = listmods[n//2:2*(n//3+1)]
group4 = listmods[2*(n//3+1):]

groupLow = group1+group2
groupHig = group3+group4
groupall = group1+group2+group3+group4

In [ ]:
run_to_module = {

    "membrane" : {
        40801 : ["M3(1)", "M3(2)"], # Mask 8, int 2250
        40808 : ["M4(1)", "M4(2)", "M6(1)", "M6(2)"], # Mask 16, int 4000
        41522 : ["M5(1)", "M5(2)"], # Mask 16, int 3500   
        41236 : ["M7(1)", "M7(2)"], # Mask 8, int 3250 
        42156 : ["M8(1)", "M8(2)"], # !! Mask 8, int 2100 -> After swapping fibers !!
    },
    "cathode" : { 
       40808 : ["C1(1)", "C1(2)", "C7(1)", "C7(2)", "C8(1)", "C8(2)"], # Mask 16, int 4000
       41519 : ["C2(1)", "C2(2)"], # Mask 8, int 3000 
       41536 : ["C3(1)", "C3(2)"], # Mask 1, 2800
       42002 : ["C4(1)", "C4(2)", "C5(1)", "C5(2)"], # Mask 16, int 2400
       40807 : ["C6(1)", "C6(2)"], # Mask 8, int 4000  
    },

    "pmt" : {
      42882 : ordered_modules_pmt 
        
    }

}
run_to_module = run_to_module[dettype]

run_to_unich = { r: [ dict_module_to_uniqch[m].channel for m in modules ] for r, modules in run_to_module.items() }
channels = [ x for v in run_to_unich.values() for x in v]

In [ ]:
import copy
import time
def select_channels(waveform: Waveform, channels: list) -> bool:
    if waveform.channel not in channels:
        return False
    return True

def create_wfset(run_to_unich, endpoint):
    nwaveforms = 1000
    wfset_full = None
    for run, channels in run_to_unich.items():
        wfset = open_processed(run, dettype, datadir, channels, [endpoint], nwaveforms=nwaveforms, verbose=True)

        if wfset_full is None:
            wfset_full = copy.deepcopy(wfset)
        else:
            wfset_full.merge(copy.deepcopy(wfset))
        print(f"Loaded run {run}")
    return wfset_full
    
start = time.time()
wfset_full = create_wfset(run_to_unich, endpoint)
end = time.time()
print(end - start)
wfset_full

In [ ]:
runBasicWfAnaNP02(wfset_full, int_ll=250, int_ul=280, amp_ll=100, amp_ul=260, threshold=25, onlyoptimal=True, configyaml="")

In [ ]:
def select_good_baseline(waveform:Waveform) -> bool:
    if waveform.analyses['std'].result['amplitude'] is np.nan:
        return False
    return True

wfset_optimal = WaveformSet.from_filtered_WaveformSet(wfset_full, select_good_baseline)
wfset_optimal

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=5000,
    adc_range_below_baseline=-50,
    adc_bins=125,
    time_bins=wfset_full.points_per_wf//2,
    filtering=8,
    share_y_scale=True,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1800,
    higth=1300,
    showplots=True
)

#detector = groupLow
#detector = groupHig
detector = groupall

#detector=["M7(1)","M7(2)"]
# detector=["C2(1)"]

plot_detectors(wfset_optimal, detector, **argsheat)


In [ ]:
cutyaml = 'cuts_template_pmt.yaml'
extractor = WaveformSelector(cutyaml)
wfset_clean = WaveformSet.from_filtered_WaveformSet(wfset_optimal, extractor.applycuts, show_progress=True)
print(f"Original waveforms: {len(wfset_optimal.waveforms)}, after cut: {len(wfset_clean.waveforms)}")

In [ ]:
argsheat['adc_range_above_baseline']=3000
argsheat['adc_range_below_baseline']=-150
argsheat['adc_bins']=125
argsheat['share_y_scale']=True
argsheat['filtering'] = 8
argsheat['return_fig'] = False
plot_detectors(wfset_clean, detector, **argsheat)

In [ ]:
argsheat['adc_range_above_baseline']=3000
argsheat['adc_range_below_baseline']=-150
argsheat['adc_bins']=125
argsheat['return_fig'] = True
figs = plot_detectors(wfset_clean, detector, **argsheat)
fig, rows, cols, title, g = figs[0]
plot_averages(fig, g)
fig.show()

In [ ]:
calibration_file = 'np02-config-v4.0.0.csv'
calibration_data = ch_read_calib(calibration_file)
template_outputdir = waffles.__path__[0] + "/np02_data/templates/templates_pmt/"

In [ ]:
# peaks_all = compute_peaks_rise_fall_ch(wfset_clean)
# for (ep, ch), vals in peaks_all.items():
#     peak_value = vals["peak_value"]

#     spe_amp = calibration_data[endpoint][ch]['SpeAmpl']
#     normalization_factor = peak_value / spe_amp

wfsetch = ChannelWsGrid.clusterize_waveform_set(wfset_clean)[endpoint]

In [ ]:
with open(template_outputdir+"cuts_used.yaml", "w") as f:
        with open(cutyaml, "r") as fcuts:
            f.write(fcuts.read())

In [ ]:
info_file = template_outputdir + f"wfdetails_{dettype}.info"

existing_data = {}

if os.path.exists(info_file):
    with open(info_file, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:  # Skip header line
            if line.strip():
                # Split by comma to get the main parts
                parts = line.split(", ")
                if len(parts) >= 3:
                    # The first part contains "Module endpoint-channel"
                    module_and_endpointch = parts[0].strip()
                    # Split to separate module from endpoint-channel
                    split_parts = module_and_endpointch.rsplit(" ", 1)
                    if len(split_parts) == 2:
                        module = split_parts[0]
                        endpoint_ch = split_parts[1]
                        run_num = int(parts[1].strip())
                        waveforms = int(parts[2].strip())
                        existing_data[(endpoint_ch, run_num)] = (module, waveforms)

# Update only channels that match the detector filter
for run_num, run_channels in run_to_unich.items():
    for ch in run_channels:
        module = dict_uniqch_to_module[strUch(endpoint, ch)]
        if module in detector:  # Only update if this module is in the detector list
            wfs = wfsetch[ch]
            endpoint_ch = f"{endpoint}-{ch}"
            existing_data[(endpoint_ch, run_num)] = (module, len(wfs.waveforms))

# Write back to file
with open(info_file, "w") as f:
    f.write("Module, endpoint-channel, run number, # of waveforms\n")
    for (endpoint_ch, run_num), (module, waveforms) in sorted(existing_data.items(), key=lambda x: x[1][0]):
        f.write(f"{module} {endpoint_ch}, {run_num}, {waveforms}\n")

In [ ]:
from waffles.np02_utils.PlotUtils import plot_averages
fig_simple = psu.make_subplots(rows=4, cols=6)
# plot_averages(fig_simple, g, calibration_data, save=False, save_dir=template_outputdir)
plot_averages(fig_simple, g, save=True, save_dir=template_outputdir)
fig_simple.show()